<a href="https://colab.research.google.com/github/raisa1521/Hands-on-Machine-Learning/blob/main/9%EC%9E%A5_%EB%B9%84%EC%A7%80%EB%8F%84_%ED%95%99%EC%8A%B5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **9장 비지도 학습**

**비지도 학습**: 레이블 y 없이 입력 특성 X를 사용하는 머신러닝

주요 작업

- **군집**: 비슷한 샘플을 클러스터로 모음
- **이상치 탐지**: 정상 데이터의 모습을 학습한 뒤 비정상 샘플을 감지
- **밀도 추정**: 데이터셋을 생성한 확률 과정의 확률 밀도 함수(PDF)를 추정

이상치 탐지에서는 비정상 샘플을 이상치, 정상 샘플을 정상치라고 함

밀도 추정에서 밀도가 매우 낮은 영역의 샘플은 이상치일 가능성이 높음

# **9.1 군집**

군집: 비슷한 샘플을 구별하여 하나의 클러스터 또는 비슷한 샘플의 그룹으로 할당하는 작업

분류와 달리 군집은 레이블을 사용하지 않는 **비지도 학습**

주요 활용

- **고객 분류**: 구매 이력이나 웹 사이트 내 행동을 기반으로 고객 그룹 구분
- **데이터 분석**: 클러스터별 데이터 분석
- **차원 축소 기법**: 각 샘플을 클러스터 친화성 벡터로 변환
- **이상치 탐지**: 모든 클러스터에 친화성이 낮은 샘플 감지
- **준지도 학습**: 동일한 클러스터에 속한 샘플에 레이블 전파
- **검색 엔진**: 비슷한 이미지가 속한 클러스터 검색
- **이미지 분할**: 비슷한 색상의 픽셀을 같은 클러스터로 모음

클러스터 친화성(affinity): 샘플이 특정 클러스터에 얼마나 잘 맞는지를 측정하는 값

클러스터에는 보편적인 정의가 없음

-> 센트로이드를 중심으로 모인 샘플, 밀집되어 연속된 영역, 계층적인 클러스터 등 알고리즘에 따라 달라짐.

### **9.1.1 K-평균**

**K-평균(K-means)**: 각 클러스터의 중심을 찾고 가장 가까운 클러스터에 샘플을 할당하는 알고리즘

- 빠르고 효율적으로 클러스터를 만듦
- 찾으려는 클러스터 개수 k를 지정해야 함
- 각 샘플의 레이블은 할당된 클러스터의 인덱스이며 분류에서 사용하는 클래스 레이블과 다름

In [2]:
from sklearn.cluster import KMeans
from sklearn.datasets import make_blobs

# [...] 생략된 부분을 실제 동작 가능한 매개변수(샘플 수 1000개, 중심점 5개)로 수정합니다.
X, y = make_blobs(n_samples=1000, centers=5, random_state=42)
                         # y는 클러스터 ID를 포함합니다
                         # 하지만 이를 사용하지 않고 예측해보겠습니다

k = 5

In [3]:
kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
y_pred = kmeans.fit_predict(X)

In [4]:
y_pred
# 결과 - array([4, 0, 1, ..., 2, 1, 0], dtype=int32)

y_pred is kmeans.labels_
# 결과 - True

True

- labels_: 훈련된 샘플의 클러스터 레이블
- cluster_centers_: 훈련된 센트로이드
- predict(): 새로운 샘플을 가장 가까운 센트로이드의 클러스터에 할당

In [5]:
kmeans.cluster_centers_

array([[-6.68630691, -6.81139455],
       [ 2.03639615,  4.26762858],
       [-2.50265159,  9.03963769],
       [-8.80798056,  7.40425704],
       [ 4.66534309,  1.87476184]])

In [6]:
import numpy as np

X_new = np.array([[0, 2], [3, 2], [-3, 3], [-3, 2.5]])
kmeans.predict(X_new)
# 결과 - array([1, 1, 2, 2], dtype=int32)

array([1, 4, 1, 1], dtype=int32)

보로노이 다이어그램: 가장 가까운 센트로이드에 따라 나뉜 클러스터의 결정 경계

K-평균은 클러스터의 크기가 많이 다르면 잘 작동하지 않음.

- **하드 군집**: 샘플을 하나의 클러스터에 할당
- **소프트 군집**: 클러스터마다 샘플에 점수를 부여

소프트 군집의 점수는 센트로이드까지의 거리 또는 유사도 점수, 친화성 점수가 될 수 있음.

transform(): 각 샘플과 모든 센트로이드 사이의 거리를 반환

In [7]:
kmeans.transform(X_new).round(2)

array([[11.06,  3.05,  7.47, 10.33,  4.67],
       [13.09,  2.46,  8.94, 12.99,  1.67],
       [10.48,  5.19,  6.06,  7.29,  7.75],
       [10.01,  5.34,  6.56,  7.6 ,  7.69]])


-> 거리가 작을수록 해당 센트로이드에 가까움

이 변환을 적용하면 각 샘플이 k개의 특성을 갖는 데이터셋이 만들어짐

-> 비선형 차원 축소 또는 다른 모델을 위한 추가 특성으로 사용 가능

### **K-평균 알고리즘**

1. k개의 센트로이드를 랜덤하게 초기화
2. 모든 샘플을 가장 가까운 센트로이드에 할당
3. 각 클러스터에 속한 샘플의 평균으로 센트로이드를 업데이트
4. 센트로이드가 더 이상 변하지 않을 때까지 반복

-> 제한된 횟수 안에 수렴하지만 항상 최적의 솔루션을 찾는 것은 아님

-> 센트로이드 초기화에 따라 지역 최적점으로 수렴할 수 있음

### **센트로이드 초기화 방법**

센트로이드의 위치를 근사하게 알고 있다면 init 매개변수에 직접 지정하고 n_init=1로 설정

In [8]:
good_init = np.array([[-3, 3], [-3, 2], [-3, 1], [-1, 2], [0, 2]])
kmeans = KMeans(n_clusters=5, init=good_init, n_init=1, random_state=42)
kmeans.fit(X)

KMeans(init=array([[-3,  3],
       [-3,  2],
       [-3,  1],
       [-1,  2],
       [ 0,  2]]),
       n_clusters=5, n_init=1, random_state=42)

또 다른 방법은 랜덤 초기화를 여러 번 실행하고 가장 좋은 솔루션을 선택하는 것

- n_init: 서로 다른 초기화로 알고리즘을 실행하는 횟수
- **이너셔(inertia)**: 각 샘플과 가장 가까운 센트로이드 사이의 제곱 거리 합

-> 이너셔가 가장 낮은 모델을 선택

In [9]:
kmeans.inertia_
# 결과 - 211.59853725816856

kmeans.score(X)
# 결과 - -211.59853725816856

-2986.675754535727

score()는 이너셔의 음숫값을 반환

-> 사이킷런의 점수는 클수록 좋은 모델을 의미하기 때문

### **K-평균++**

**K-평균++**: 다른 센트로이드와 거리가 먼 샘플을 선택하는 초기화 방법

1. 데이터셋에서 랜덤으로 첫 번째 센트로이드 선택
2. 이미 선택된 가장 가까운 센트로이드까지의 제곱 거리에 비례하는 확률로 다음 센트로이드 선택
3. k개의 센트로이드를 선택할 때까지 반복

-> 최적이 아닌 솔루션으로 수렴할 가능성을 줄임

-> KMeans 클래스에서 기본적으로 사용하는 초기화 방법

### **K-평균 속도 개선과 미니배치 K-평균**

**엘칸 알고리즘**

- 삼각 부등식과 센트로이드 사이 거리의 하한선·상한선을 활용
- 불필요한 거리 계산을 줄여 속도를 높임
- 데이터셋에 따라 오히려 느려질 수도 있음
- algorithm="elkan"으로 지정 가능

**미니배치 K-평균**

전체 데이터셋 대신 반복마다 미니배치를 사용해 센트로이드를 조금씩 이동

- 일반 K-평균보다 빠름
- 메모리에 들어가지 않는 대량의 데이터셋에도 적용 가능
- 이너셔는 일반적으로 조금 더 나쁨

In [10]:
from sklearn.cluster import MiniBatchKMeans

minibatch_kmeans = MiniBatchKMeans(n_clusters=5, random_state=42, n_init=3)
minibatch_kmeans.fit(X)

MiniBatchKMeans(n_clusters=5, n_init=3, random_state=42)

MiniBatchKMeans는 memmap이나 partial_fit()을 이용할 수 있음

일반 K-평균보다 약 3.5배 빠르게 훈련됨

### **최적의 클러스터 개수 찾기**

클러스터 개수 k를 잘못 설정하면

- k가 너무 작음 -> 서로 다른 클러스터가 합쳐짐
- k가 너무 큼 -> 하나의 클러스터가 여러 개로 나뉨

**이너셔를 이용한 엘보 방법**

이너셔는 k가 증가할수록 감소하므로 이너셔가 가장 작은 k를 단순히 선택할 수 없음

-> k에 따른 이너셔 그래프에서 감소 속도가 크게 둔화되는 엘보를 찾음

교재 예제에서는 k=4가 엘보에 해당

### **실루엣 점수**

실루엣 점수: 모든 샘플의 실루엣 계수의 평균

실루엣 계수

$s=\frac{b-a}{\max(a,b)}$

- a: 동일한 클러스터 내 다른 샘플까지의 평균 거리
- b: 가장 가까운 다른 클러스터까지의 평균 거리

실루엣 계수의 범위는 -1부터 +1

- +1에 가까움 -> 클러스터 내부에 잘 속하고 다른 클러스터와 멀리 떨어짐
- 0에 가까움 -> 클러스터 경계에 위치
- -1에 가까움 -> 잘못된 클러스터에 할당되었을 가능성이 높음

In [11]:
from sklearn.metrics import silhouette_score

silhouette_score(X, kmeans.labels_)
# 결과 - 0.65517642572828

np.float64(0.6213223538157563)

**실루엣 다이어그램**

- 그래프의 높이 -> 클러스터의 샘플 수
- 그래프의 너비 -> 정렬된 실루엣 계수
- 수직 파선 -> 평균 실루엣 점수

클러스터의 샘플 대부분이 수직 파선의 왼쪽에 있으면 나쁜 클러스터

교재의 예제에서는 k=4의 평균 실루엣 점수가 더 높지만 하나의 클러스터가 매우 큼

반면 k=5에서는 클러스터 크기가 비슷하므로 k=5를 선택하는 것이 좋음.

### **9.1.2 K-평균의 한계**

K-평균의 장점

- 빠른 속도
- 확장 용이

한계

- 최적이 아닌 솔루션을 피하려면 여러 번 실행해야 함.
- 클러스터 개수 k를 미리 지정해야 함.
- 클러스터의 크기, 밀집도, 방향이 서로 다르거나 원형이 아니면 잘 작동하지 않음.

-> 타원형 클러스터에서는 가우스 혼합 모델이 더 잘 작동할 수 있음

K-평균 실행 전 **특성 스케일**을 맞추는 것이 중요

-> 특성 스케일을 맞추면 일반적으로 K-평균에 도움이 되지만 모든 클러스터가 잘 구분된다는 보장은 없음

### **9.1.3 군집을 사용한 이미지 분할**

**이미지 분할(image segmentation)**: 이미지를 여러 개의 세그먼트로 분할하는 작업

- **색상 분할**: 동일한 색상의 픽셀을 같은 세그먼트에 할당
- **시맨틱 분할**: 동일한 종류의 물체에 속한 모든 픽셀을 같은 세그먼트에 할당
- **인스턴스 분할**: 개별 객체에 속한 모든 픽셀을 같은 세그먼트에 할당

여기서는 K-평균을 사용한 **색상 분할**을 수행

In [13]:
#자동으로 고침(다시 확인하)
import PIL
import urllib.request

# 샘플 이미지를 다운로드하여 filepath에 저장합니다.
url = "https://github.com/rickiepark/handson-ml2/raw/master/images/unsupervised_learning/ladybug.png"
filepath = "ladybug.png"
urllib.request.urlretrieve(url, filepath)

image = np.asarray(PIL.Image.open(filepath))

In [14]:
image.shape
# 결과 - (533, 800, 3)

(533, 800, 3)

이미지는 3D 배열로 표현

- 첫 번째 차원 -> 높이
- 두 번째 차원 -> 너비
- 세 번째 차원 -> 색상 채널

각 픽셀은 빨강·초록·파랑(RGB)의 강도를 포함하는 3D 벡터

각 채널의 값은 0~ 255

In [15]:
X = image.reshape(-1, 3)
kmeans = KMeans(n_clusters=8, random_state=42, n_init=10).fit(X)
segmented_img = kmeans.cluster_centers_[kmeans.labels_]
segmented_img = segmented_img.reshape(image.shape)

1. 이미지를 RGB 색상 리스트로 변환
2. K-평균을 사용해 8개의 클러스터 생성
3. 각 픽셀을 클러스터의 평균 색상으로 변경
4. 원래 이미지 크기로 복원

클러스터 수가 너무 적으면 크기가 작은 물체의 색상이 주변 색상과 합쳐질 수 있음

-> K-평균은 비슷한 크기의 클러스터를 만드는 경향이 있기 때문